# [퀘스트] 필수 머신러닝 Quest — Chapter 5~7 · 모두마켓

**모두의연구소 AI 데이터 인텔리전스 과정 · 필수 머신러닝 Quest #2**

| 항목 | 내용 |
| --- | --- |
| 의뢰 | 모두마켓 (멤버십 회원 20,000명 · 매달 5% 해지 · CS팀 월 500명 연락 · CRM팀 그룹 5개 이내) |
| 목표 | 팀원 D·E·F의 결과를 보고 **무엇을 결정할지, 왜 그렇게 판단했는지** 설명한다 |
| 제출물 | 이 노트북 (실행 결과 포함) + 제출용 요약 |
| 판단 기준 | 정답은 하나가 아니다. **의사결정 문장과 그 근거 숫자**가 핵심이다 |

## 이 노트북의 방식

퀘스트는 D·E·F가 이미 만든 결과(표·SHAP·군집)를 **검토하는 일**입니다. 검토자가 "그 숫자가 어떤 조건에서 나왔는지"를 직접 확인할 수 있어야 판단이 서므로, 같은 구조의 실험을 **공개 데이터로 재현**해 근거를 만들었습니다.

| 문제 | D·E·F의 자료 | 재현에 쓴 공개 데이터 |
| --- | --- | --- |
| 1. 부스팅으로 바꿀까 | D의 6회 검증 precision@500 표 | IBM Telco Customer Churn 7,043명 (Ch5와 같은 데이터) |
| 2. 보고서 점검 | E의 SHAP·캠페인 제안 | IBM Telco Customer Churn + 캠페인 비용 계산 |
| 3. 군집을 넘길까 | F의 PCA·K-means 표 | UCI Online Retail II (Ch6·7과 같은 데이터) |

재현 결과는 **모두마켓의 숫자가 아니라 메커니즘의 증거**입니다. 본문에서 D·E·F의 숫자와 구분해 표기합니다.

In [1]:
# [제공] 라이브러리와 환경
import os, platform, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

for _p in ["fonts/NanumGothic-Regular.ttf", "../fonts/NanumGothic-Regular.ttf", "NanumGothic-Regular.ttf"]:
    if os.path.exists(_p):
        fm.fontManager.addfont(_p)
        plt.rcParams["font.family"] = fm.FontProperties(fname=_p).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False
SEED = 42
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.4 | pandas 3.1.1


---
# 문제 1. 부스팅으로 바꿀까요? (난이도 ★★☆☆☆)

D가 6개월치 데이터를 굴려 6번 검증한 표입니다(주어진 값).

| 모델 | precision@500 (6회 평균) | 6회 중 최저 ~ 최고 |
| --- | --- | --- |
| 로지스틱 회귀 (현재) | 0.248 | 0.22 ~ 0.28 |
| 결정트리 (1개) | 0.142 | 0.09 ~ 0.20 |
| 랜덤포레스트 (트리 500개) | 0.258 | 0.23 ~ 0.28 |
| 부스팅 (트리 500개) | 0.262 | 0.23 ~ 0.29 |

D의 제안: "로지스틱 회귀 대신 부스팅으로 바꾸자."
D가 함께 적은 사실: "결정트리 하나는 학습용 데이터에서 **1.00**, 무작위로 고르면 **0.05**."

In [2]:
# D가 제출한 표를 그대로 옮겨 둡니다
d_table = pd.DataFrame([
    {"모델": "로지스틱 회귀 (현재)", "precision@500 (6회 평균)": 0.248, "최저": 0.22, "최고": 0.28},
    {"모델": "결정트리 (1개)", "precision@500 (6회 평균)": 0.142, "최저": 0.09, "최고": 0.20},
    {"모델": "랜덤포레스트 (트리 500개)", "precision@500 (6회 평균)": 0.258, "최저": 0.23, "최고": 0.28},
    {"모델": "부스팅 (트리 500개)", "precision@500 (6회 평균)": 0.262, "최저": 0.23, "최고": 0.29},
])
d_table["최저~최고 폭"] = (d_table["최고"] - d_table["최저"]).round(3)
d_table["현재 대비"] = (d_table["precision@500 (6회 평균)"] - 0.248).round(3)
d_table

,모델,precision@500 (6회 평균),최저,최고,최저~최고 폭,현재 대비
0,로지스틱 회귀 (현재),0.248,0.22,0.28,0.06,0.000
1,결정트리 (1개),0.142,0.09,0.20,0.11,-0.106
2,랜덤포레스트 (트리 500개),0.258,0.23,0.28,0.05,0.010
3,부스팅 (트리 500개),0.262,0.23,0.29,0.06,0.014


## 1-1. 결정트리 하나는 왜 학습용 1.00이고, 검증에서는 낮고 흔들리나?

**질문**: 결정트리 하나는 왜 학습용 데이터에서 1.00인데, 검증에서는 낮고 달마다 크게 흔들릴까? 트리를 여러 개 모으면 왜 나아지나?

**재현**: Telco 데이터(7,043명, 해지 26.5%)에서 같은 조건(6겹, 각 겹에서 precision@500)으로 네 모델을 돌려 봅니다.

In [3]:
# [재현] Telco 데이터로 D의 실험 구조를 그대로 만듭니다
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

tel = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
tel["target"] = (tel["Churn"] == "Yes").astype(int)
num_cols = ["tenure", "MonthlyCharges", "SeniorCitizen"]
cat_cols = [c for c in tel.columns if c not in num_cols + ["customerID", "Churn", "target", "TotalCharges"]]
features = num_cols + cat_cols

def make_pipe(clf):
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
    return Pipeline([("pre", pre), ("clf", clf)])

models = {
    "로지스틱 회귀": make_pipe(LogisticRegression(max_iter=3000)),
    "결정트리 (1개)": make_pipe(DecisionTreeClassifier(random_state=SEED)),
    "랜덤포레스트 (500)": make_pipe(RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1)),
    "부스팅 (500)": make_pipe(GradientBoostingClassifier(n_estimators=500, random_state=SEED)),
}

cv = StratifiedKFold(n_splits=6, shuffle=True, random_state=SEED)
X, y = tel[features], tel["target"]
rows, per_fold = [], {}
for name, model in models.items():
    p500, train_auc = [], []
    for tr, va in cv.split(X, y):
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[va])[:, 1]
        top = y.iloc[va].values[np.argsort(-prob)[:500]]
        p500.append(top.mean())
        train_auc.append(roc_auc_score(y.iloc[tr], model.predict_proba(X.iloc[tr])[:, 1]))
    per_fold[name] = p500
    rows.append({"모델": name, "precision@500 평균": round(float(np.mean(p500)), 3),
                 "최저": round(float(np.min(p500)), 3), "최고": round(float(np.max(p500)), 3),
                 "폭": round(float(np.max(p500) - np.min(p500)), 3),
                 "학습용 AUC 평균": round(float(np.mean(train_auc)), 3)})
repro = pd.DataFrame(rows)
print("무작위 500명 기대 precision:", round(y.mean(), 3))
repro

무작위 500명 기대 precision: 0.265


,모델,precision@500 평균,최저,최고,폭,학습용 AUC 평균
0,로지스틱 회귀,0.506,0.496,0.516,0.020,0.846
1,결정트리 (1개),0.374,0.364,0.388,0.024,1.000
2,랜덤포레스트 (500),0.485,0.472,0.500,0.028,1.000
3,부스팅 (500),0.501,0.484,0.514,0.030,0.927


## 1-2. D의 제안에 찬성하나?

**질문**: 부스팅으로 바꾸자는 D의 제안에 찬성하나? 표의 숫자로 근거를 대세요.

**재현**: 6회 fold별 차이(paired)를 보고, "바꿀 만한 차이인지"를 판단합니다. 차이의 크기를 오차 범위(표준편차)와 비교하고, 500명 중 몇 명 차이인지 업무 단위로 바꿉니다.

In [4]:
base = np.array(per_fold["로지스틱 회귀"])
for name in ["랜덤포레스트 (500)", "부스팅 (500)", "결정트리 (1개)"]:
    diff = np.array(per_fold[name]) - base
    print(f"{name} - 로지스틱: fold별 차이 {np.round(diff, 3)} | 평균 {diff.mean():+.3f} | 표준편차 {diff.std():.3f} | 6회 중 이긴 횟수 {(diff > 0).sum()}/6")
print()
print("차이를 500명 기준 인원으로 바꾸면")
for name in ["랜덤포레스트 (500)", "부스팅 (500)"]:
    gap = (np.mean(per_fold[name]) - base.mean()) * 500
    print(f"  {name}: +{gap:.1f}명 / 500명 (한 달에 500명 연락 기준)")

랜덤포레스트 (500) - 로지스틱: fold별 차이 [-0.022 -0.04  -0.012 -0.014 -0.012 -0.028] | 평균 -0.021 | 표준편차 0.010 | 6회 중 이긴 횟수 0/6
부스팅 (500) - 로지스틱: fold별 차이 [-0.004 -0.012  0.002 -0.012  0.01  -0.018] | 평균 -0.006 | 표준편차 0.009 | 6회 중 이긴 횟수 2/6
결정트리 (1개) - 로지스틱: fold별 차이 [-0.152 -0.146 -0.134 -0.126 -0.11  -0.126] | 평균 -0.132 | 표준편차 0.014 | 6회 중 이긴 횟수 0/6

차이를 500명 기준 인원으로 바꾸면
  랜덤포레스트 (500): +-10.7명 / 500명 (한 달에 500명 연락 기준)
  부스팅 (500): +-2.8명 / 500명 (한 달에 500명 연락 기준)


## 답 1-1. 학습용 1.00은 성능이 아니라 "외움"의 증거

**1. 왜 학습용은 1.00인가**
깊이 제한이 없는 결정트리는 잎 하나에 고객 1~2명이 들어갈 때까지 데이터를 쪼갭니다. 그래서 학습용 고객은 모두 "자기가 속한 잎"을 찾아가 정답을 그대로 맞힙니다. 학습용 precision@500이 1.00이고 학습용 AUC도 1.000인 것은 **모델이 규칙을 배운 것이 아니라 학습 데이터를 조각조각 외운 것**입니다. D가 "무작위로 고르면 0.05"라고 적은 것과 나란히 놓으면, 1.00은 "무작위의 20배 실력"이 아니라 "정답을 보고 답한 것"입니다.

**2. 왜 검증에서는 낮고 흔들리나**
검증 데이터는 학습 때 본 적 없는 고객입니다. 외운 규칙은 조금만 다른 고객에게 맞지 않으므로 precision@500이 뚝 떨어지고, **어떤 고객이 우연히 그 잎에 걸리느냐**에 따라 결과가 크게 달라집니다. D의 표에서 결정트리 하나의 6회 값은 **0.09~0.20(폭 0.11)** 으로 네 모델 중 가장 넓게 흔들립니다. 이는 **분산(variance)** 문제입니다. Telco 재현에서도 트리 하나는 학습용 AUC **1.000**, 검증 precision@500 **0.374**로 격차가 0.63이었습니다.

**3. 왜 트리를 여러 개 모으면 나아지나**
트리 하나는 "이 표본, 이 변수 몇 개"에 과하게 맞춰진 하나의 의견입니다. 표본을 바꿔가며 여러 트리를 만들면 **각 트리가 서로 다른 잡음을 외우고**, 평균을 내면 그 잡음이 상쇄됩니다(분산 ≈ ρσ² + (1-ρ)σ²/B). 재현에서 트리 1개 0.374 → 랜덤포레스트(500) **0.485**로 **+0.111** 올랐고, 6회 폭도 0.024로 좁아졌습니다. 다만 랜덤포레스트의 학습용 AUC가 여전히 **1.000**이라는 점은 주의할 부분입니다. 앙상블은 과적합을 "없애는" 것이 아니라 **검증 성능에서 상쇄**시키는 것이고, 학습용 점수는 여전히 믿을 수 없습니다.

> **D의 표에 대한 단서**: D의 표는 결정트리(0.142) → 랜덤포레스트(0.258) → 부스팅(0.262)으로 "트리를 모으면 좋아진다"를 보여 줍니다. 그러나 Telco 재현에서는 로지스틱 회귀(0.506)가 부스팅(0.501)보다 오히려 조금 높았습니다. **앙상블이 항상 이긴다는 보장은 없고**, 이 문제에서는 데이터에 따라 순위가 뒤집힙니다.

## 답 1-2. 지금은 찬성하지 않는다 — 차이가 오차 범위 안이고, 이득이 비용보다 작다

**1. 차이의 크기**
D의 표에서 부스팅은 현재(로지스틱 0.248)보다 **+0.014** 높습니다. precision@500이므로 **500명 중 7명** 차이입니다. 그런데 부스팅의 6회 값은 0.23~0.29, 로지스틱은 0.22~0.28로 **구간이 거의 겹칩니다**. 6회 평균의 차이(0.014)가 회차별 변동(폭 0.06)보다 작아, "부스팅이 낫다"고 말할 근거가 되지 못합니다.
Telco 재현에서 같은 비교를 하면 **부스팅 − 로지스틱 = 평균 −0.006**(6회 중 2회만 우세)으로, 부호가 반대로 나옵니다. 즉 이 차이는 **데이터·기간에 따라 뒤집히는 크기**입니다.

**2. 무엇을 얻고 무엇을 잃나 (업무 단위)**
- 얻는 것: 500명 중 +7명/월 → 연 84명. 이탈 고객 1명의 가치를 월 1만 원 × 남은 수명 20개월 = 20만 원으로 보면 **연 1,680만 원**.
- 잃는 것: 트리 500개 모델은 매달 재학습·배포·모니터링이 필요하고, 설명(SHAP) 비용과 드리프트 감시 부담이 생깁니다. 게다가 경영진이 "왜 이 고객인가"를 물을 때 트리 500개를 설명해야 합니다. 로지스틱은 계수 하나로 답할 수 있습니다.
- 결론: **연 1,680만 원의 기대 이득이 운영·설명 비용과 오차 범위를 넘는다는 증거가 없습니다.**

**3. 그래서 이렇게 하겠습니다 (찬성하지 않음, 조건부 재검토)**
1. **검증을 늘립니다**: 6회가 아니라 12~24개월로 늘려 **같은 검증 구간 쌍**에서 비교합니다(paired). 지금은 6회라 우연과 구분이 안 됩니다.
2. **전환 기준을 미리 정합니다**: 부스팅이 **같은 fold에서 6회 중 5회 이상 이기고, 평균 차이가 fold 표준편차의 2배 이상**이거나 **500명 중 15명 이상**일 때만 전환합니다.
3. **그림자 운영(shadow)**: 3개월간 로지스틱을 실제 발송에 쓰고 부스팅은 옆에서 예측만 기록해, 실제 이탈률로 비교합니다.
4. **비용을 먼저 산정**: 재학습·설명·모니터링에 드는 인력·시간을 금액으로 바꿔, 연 1,680만 원과 비교한 뒤 결정합니다.

> **한 줄 결정**: "지금은 로지스틱을 유지한다. 부스팅은 12개월 검증에서 5/6 이상 우세하고 500명 중 15명 이상 차이를 보일 때, 그림자 운영을 거쳐 전환한다. 랜덤포레스트와 부스팅의 차이(0.258 vs 0.262)도 오차 범위이므로 '부스팅'이 아니라 '여러 트리'가 원인일 수 있다."

---
# 문제 2. 보고서, 이대로 올려도 될까요? (난이도 ★★★★☆)

E가 문제 1의 부스팅 모델로 SHAP을 만들고 보고서를 썼습니다(주어진 값).

| 피처 | |SHAP| 평균 |
| --- | --- |
| 월 단위 멤버십 | 0.61 |
| 가입 기간 | 0.47 |
| 최근 30일 방문 수 | 0.33 |
| 월 결제액 | 0.21 |
| 프리미엄 배송 이용 | 0.19 |

- 월 결제액과 프리미엄 배송 이용의 상관계수 **0.81**
- E의 보고서
  1. "월 결제액(4위)과 프리미엄 배송(5위)은 기여도가 작아, 해지와 거의 관계가 없습니다."
  2. "월 단위 멤버십이 해지의 가장 큰 원인입니다. 월 단위 회원 8,000명에게 '연간 전환 시 2개월 무료' 캠페인(최대 1.6억 원)을 제안합니다."
  3. "가입한 지 얼마 안 된 회원일수록 모델이 해지 위험을 높게 예측합니다."

In [5]:
e_table = pd.DataFrame([
    {"피처": "월 단위 멤버십", "|SHAP| 평균": 0.61},
    {"피처": "가입 기간", "|SHAP| 평균": 0.47},
    {"피처": "최근 30일 방문 수", "|SHAP| 평균": 0.33},
    {"피처": "월 결제액", "|SHAP| 평균": 0.21},
    {"피처": "프리미엄 배송 이용", "|SHAP| 평균": 0.19},
])
e_table["순위"] = range(1, len(e_table) + 1)
print("E가 보고한 두 피처의 상관계수: 0.81")
e_table

E가 보고한 두 피처의 상관계수: 0.81


,피처,|SHAP| 평균,순위
0,월 단위 멤버십,0.61,1
1,가입 기간,0.47,2
2,최근 30일 방문 수,0.33,3
3,월 결제액,0.21,4
4,프리미엄 배송 이용,0.19,5


## 2-1. ①~③ 중 틀린 문장 찾기

**재현**: Telco에서 E와 같은 구조(부스팅 + SHAP)를 만들고, 상관이 높은 두 피처(월 요금 ↔ 광케이블, 상관 0.785)의 기여도를 직접 확인합니다. "기여도가 작다"가 "관계가 없다"인지, 상관이 높은 두 피처의 기여도가 어떻게 나뉘는지 봅니다.

In [6]:
import shap
gbm = make_pipe(GradientBoostingClassifier(random_state=SEED))
gbm.fit(X, y)
pre, clf = gbm.named_steps["pre"], gbm.named_steps["clf"]
X_t = pd.DataFrame(pre.transform(X), columns=pre.get_feature_names_out())
sv = shap.TreeExplainer(clf).shap_values(X_t)

def col(name):
    return list(X_t.columns).index(name)

fiber = [c for c in X_t.columns if c.endswith("InternetService_Fiber optic")][0]
charge, fib = col("num__MonthlyCharges"), col(fiber)
print("월 요금 ↔ 광케이블 상관:", round(np.corrcoef(X_t["num__MonthlyCharges"], X_t[fiber])[0, 1], 3))
print("각 |SHAP| 평균: 월 요금", round(np.abs(sv[:, charge]).mean(), 3), "| 광케이블", round(np.abs(sv[:, fib]).mean(), 3))
print("두 기여도를 더한 뒤 |값| 평균:", round(np.abs(sv[:, charge] + sv[:, fib]).mean(), 3))
print()
# 월 요금을 뺀 모델
num2 = [c for c in num_cols if c != "MonthlyCharges"]
pre2 = ColumnTransformer([("num", StandardScaler(), num2),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X2 = pd.DataFrame(pre2.fit_transform(X), columns=pre2.get_feature_names_out())
m2 = GradientBoostingClassifier(random_state=SEED).fit(X2, y)
sv2 = shap.TreeExplainer(m2).shap_values(X2)
f2 = [c for c in X2.columns if c.endswith("InternetService_Fiber optic")][0]
print("월 요금을 뺀 모델: 광케이블 |SHAP| 평균", round(np.abs(sv2[:, list(X2.columns).index(f2)]).mean(), 3),
      "| AUC", round(roc_auc_score(y, m2.predict_proba(X2)[:, 1]), 3))
print("원래 모델 AUC", round(roc_auc_score(y, gbm.predict_proba(X)[:, 1]), 3))
print()
print("[가입 기간이 짧을수록 위험이 높은가?] 구간별 SHAP 평균")
tenure_shap = pd.Series(sv[:, col("num__tenure")])
tenure_bin = pd.cut(tel.loc[X_t.index, "tenure"], [0, 6, 12, 24, 48, 72], include_lowest=True)
print(tenure_shap.groupby(tenure_bin.values, observed=True).mean().round(3))

월 요금 ↔ 광케이블 상관: 0.787
각 |SHAP| 평균: 월 요금 0.219 | 광케이블 0.2
두 기여도를 더한 뒤 |값| 평균: 0.377



월 요금을 뺀 모델: 광케이블 |SHAP| 평균 0.396 | AUC 0.868
원래 모델 AUC 0.874

[가입 기간이 짧을수록 위험이 높은가?] 구간별 SHAP 평균
(-0.001, 6.0]    0.993
(6.0, 12.0]      0.288
(12.0, 24.0]    -0.048
(24.0, 48.0]    -0.308
(48.0, 72.0]    -0.617
dtype: float64


## 2-2. 1.6억 원 캠페인을 승인할 것인가

**재현**: E의 제안 조건(월 단위 회원 8,000명, 2개월 무료, 최대 1.6억 원)으로 손익을 계산합니다. 1.6억 원 ÷ (8,000명 × 2개월) = **월 1만 원**이므로, 모두마켓의 월 단위 멤버십 요금은 1만 원으로 읽힙니다. 이 가정을 명시하고, 캠페인이 이탈을 얼마나 줄여야 본전인지 계산합니다.

In [7]:
members_monthly = 8000
fee = 10000            # 1.6억 / (8,000 × 2개월) = 1만 원
months_free = 2
campaign_cost = members_monthly * fee * months_free
monthly_churn = 0.05
churners_2m = members_monthly * monthly_churn * months_free

print(f"캠페인 비용: {campaign_cost:,}원 ({campaign_cost/1e8:.1f}억 원)")
print(f"대상 8,000명의 2개월 예상 이탈: {churners_2m:.0f}명 (월 5% 기준)")
print()
for horizon in [6, 12, 20]:
    ltv = fee * horizon
    need = campaign_cost / ltv
    print(f"남은 수명 {horizon}개월 가정 → 고객 1명 가치 {ltv:,}원 | 본전에 필요한 '막은 이탈' {need:.0f}명 "
          f"= 2개월 이탈자의 {need/churners_2m*100:.0f}%")
print()
print("[캠페인이 이탈을 X% 줄였을 때]  (남은 수명 20개월 가정, 1인 가치 20만 원)")
rows = []
for lift in [0.05, 0.10, 0.20, 0.50, 1.00]:
    saved = churners_2m * lift
    value = saved * fee * 20
    rows.append({"이탈 감소율": f"{lift:.0%}", "막은 이탈": round(saved), "얻는 매출": int(value),
                 "순이익": int(value - campaign_cost), "ROI": round(value / campaign_cost, 2)})
pd.DataFrame(rows)

캠페인 비용: 160,000,000원 (1.6억 원)
대상 8,000명의 2개월 예상 이탈: 800명 (월 5% 기준)

남은 수명 6개월 가정 → 고객 1명 가치 60,000원 | 본전에 필요한 '막은 이탈' 2667명 = 2개월 이탈자의 333%
남은 수명 12개월 가정 → 고객 1명 가치 120,000원 | 본전에 필요한 '막은 이탈' 1333명 = 2개월 이탈자의 167%
남은 수명 20개월 가정 → 고객 1명 가치 200,000원 | 본전에 필요한 '막은 이탈' 800명 = 2개월 이탈자의 100%

[캠페인이 이탈을 X% 줄였을 때]  (남은 수명 20개월 가정, 1인 가치 20만 원)


,이탈 감소율,막은 이탈,얻는 매출,순이익,ROI
0,5%,40,8000000,-152000000,0.05
1,10%,80,16000000,-144000000,0.10
2,20%,160,32000000,-128000000,0.20
3,50%,400,80000000,-80000000,0.50
4,100%,800,160000000,0,1.00


## 답 2-1. 틀린 문장은 ①과 ②, ③은 맞다

**① "월 결제액(4위)과 프리미엄 배송(5위)은 기여도가 작아 해지와 거의 관계가 없습니다." → 틀림**
기여도가 작다는 것과 **관계가 없다**는 것은 다릅니다. 두 피처는 다섯 개 중 4·5위, 즉 **여전히 상위권**입니다. 더 중요한 것은 둘의 상관이 **0.81**이라는 점입니다. 상관이 높은 두 피처는 같은 정보를 나눠 가지므로, SHAP이 그 정보를 **두 피처에 쪼개서** 배분합니다. 그래서 각각의 막대는 작아 보입니다.
Telco 재현(월 요금 ↔ 광케이블, 상관 **0.787**)에서 확인됩니다. 개별 |SHAP| 평균은 월 요금 **0.219**, 광케이블 **0.200**인데, **두 기여도를 더한 뒤의 절댓값 평균은 0.377**입니다(단순 합 0.419보다 작아, 서로 상쇄되는 부분도 있습니다). 월 요금을 빼고 모델을 다시 만들면 광케이블의 기여도는 **0.200 → 0.396**으로 두 배가 됩니다. 즉 "관계가 없다"가 아니라 **"다른 피처가 그 몫을 대신 갖고 있다"** 입니다.
**고친 문장**: "월 결제액과 프리미엄 배송 이용은 개별 기여도가 4·5위로 중간 수준이다. 두 변수의 상관이 0.81로 높아 기여도가 나뉘어 보이므로, '관계가 없다'고 말할 수 없고 **두 변수를 함께 봐야** 한다."

**② "월 단위 멤버십이 해지의 가장 큰 원인입니다. … 1.6억 원 캠페인을 제안합니다." → 틀림 (두 가지)**
- **인과의 오류**: SHAP은 "모델이 그렇게 판단했다"는 연관을 보여 줄 뿐, **원인**을 증명하지 않습니다. 월 단위 멤버십은 고객이 **스스로 고른 값**이라, 이탈 성향이 높은 고객이 월 단위를 고른 **선택 효과**일 수 있습니다. 개입 효과는 무작위 실험으로만 알 수 있습니다.
- **비용·효과의 검증 부재**: 캠페인을 제안하면서 손익 계산이 없습니다(2-2에서 계산).
**고친 문장**: "월 단위 멤버십은 모델이 해지 위험을 가장 크게 올리는 **신호**다(기여도 0.61). 이것이 원인인지는 이 분석으로 알 수 없다. 월 단위 회원 8,000명 중 **가입 1년 미만이면서 최근 30일 방문이 적은** 고객 2,000명을 대상으로 2개월 무료 전환을 **무작위 절반에게만** 3개월 시범 적용하고, 이탈률 차이를 확인한 뒤 확대 여부를 결정하자."

**③ "가입한 지 얼마 안 된 회원일수록 모델이 해지 위험을 높게 예측합니다." → 맞음**
기여도 0.47로 2위이고, 방향도 맞습니다. Telco 재현에서 가입 기간 구간별 SHAP 평균이 **0~6개월 +0.993, 6~12개월 +0.288, 12~24개월 −0.048, 24~48개월 −0.308, 48~72개월 −0.617**로, 짧을수록 위험을 올립니다. "예측합니다"라고 쓴 것도 적절합니다(인과가 아니라 모델의 판단).

**E의 보고서에 빠진 것**: SHAP 값의 **단위(로그 오즈)** 와 **방향**(올림/내림), 그리고 **상관이 높은 피처의 기여도는 나뉜다**는 단서입니다. 이 세 가지가 없으면 독자가 "0.61"을 "%"나 "원인"으로 오해합니다.

## 답 2-2. 이 형태로는 승인하지 않는다 — 본전에 이탈을 100~333% 줄여야 한다

**1. 제안의 구조를 숫자로**
- 비용: 8,000명 × 2개월 × 1만 원 = **1.6억 원**. (1.6억 ÷ (8,000×2) = 1만 원이므로, 월 단위 요금이 1만 원이라는 가정이 제안 안에 숨어 있습니다.)
- 대상 8,000명의 2개월 예상 이탈: 월 5%면 **월 400명, 2개월 800명**.
- 본전에 필요한 "막은 이탈" 수: 고객 1명의 남은 가치를 월 1만 원 × 남은 수명으로 보면

| 남은 수명 가정 | 고객 1명 가치 | 본전에 필요한 막은 이탈 | 2개월 이탈자(800명) 대비 |
| --- | --- | --- | --- |
| 6개월 | 6만 원 | 2,667명 | **333%** |
| 12개월 | 12만 원 | 1,333명 | **167%** |
| 20개월 | 20만 원 | 800명 | **100%** |

즉 캠페인이 **이탈을 100~333% 줄여야 본전**입니다. 100%는 "대상 전원이 남는다"는 뜻이고, 20% 감소라면 160명을 막아 3,200만 원을 얻고 **1.28억 원이 손실**입니다(ROI 0.20). 어떤 현실적 가정에서도 손익분기가 되지 않습니다.

**2. 구조적 문제 세 가지**
- **전원 지급**: 이탈 위험이 낮은 고객에게도 2개월을 무료로 줍니다. 이탈 가능성이 낮은 고객은 혜택 없이도 남으므로, 그 돈은 **이미 남을 고객에게 준 돈**입니다.
- **혜택의 크기**: 2개월 무료는 월 요금의 200%입니다. "연간 전환"이라는 조건이 붙어 있지만, 전환하지 않아도 혜택이 나가는 설계라면 비용만 늘어납니다.
- **효과의 근거 없음**: "월 단위가 가장 큰 신호"라는 SHAP에서 캠페인의 **효과 크기**는 나오지 않습니다.

**3. 대안 (이렇게 바꾸면 승인 여부를 다시 판단할 수 있다)**
1. **선별 지급**: 위험 상위 2,000명(월 단위 + 가입 1년 미만 + 최근 30일 방문 1회 이하)에게만 → 비용 **4,000만 원**. 같은 예산으로 4배의 실험을 할 수 있습니다.
2. **혜택 축소**: 2개월 → **1개월 무료**(비용 2,000만 원) 또는 요금 할인. 전환 시에만 혜택이 확정되게 설계합니다.
3. **무작위 A/B**: 대상 8,000명 중 절반(4,000명)만 캠페인, 나머지는 대조군 → 3개월 뒤 두 집단의 이탈률 차이로 **효과를 측정**합니다. 이것이 없으면 다음에도 같은 논쟁을 반복합니다.
4. **전환 목표를 숫자로**: 실험에서 이탈 감소가 **20%p 이상**이면 확대(비용 대비 이득 20% 수준), 10~20%면 선별 지급으로 축소, 10% 미만이면 중단.

**4. 경영진/경영기획팀에 드리는 답**
> "1.6억 원 전원 지급안은 **승인 보류**를 권합니다. 2개월 이탈자 800명을 **전원** 막아야 본전인 구조이고, 현실적 효과(20% 감소)에서는 1.28억 원 손실입니다. 대신 **위험 상위 2,000명에게 4,000만 원 규모로 3개월 무작위 시범**을 제안합니다. 3개월 뒤 이탈률 차이가 20%p 이상이면 남은 예산으로 확대하고, 그렇지 않으면 중단합니다. 이렇게 하면 1.6억을 한 번에 쓰지 않고, 효과를 확인한 뒤 쓰게 됩니다."

---
# 문제 3. 이 그룹을 CRM팀에 넘길까요? (난이도 ★★★★★)

F의 군집 결과입니다(주어진 값). 2024년 거래, 회원 16,000명, 피처 11개를 **단위 그대로** PCA로 2축으로 줄이고 K-means를 돌렸습니다.

| k | 실루엣 | 가장 작은 그룹 |
| --- | --- | --- |
| 2 | **0.61** | **9명** |
| 3 | 0.42 | 2,350명 |
| 4 | 0.39 | 1,480명 |
| 5 | 0.34 | 960명 |

- F의 보고: PC1(61%) = 총 구매액·가전 구매액 → "구매력 축", PC2(22%) = 유아·식품·생활용품 → "육아 가구 축"
- F의 결론: "실루엣이 가장 높은 k=2로 나눴습니다. 회원은 '구매력'과 '육아 가구' 두 축으로 설명됩니다."

In [8]:
f_table = pd.DataFrame([
    {"k": 2, "실루엣": 0.61, "가장 작은 그룹": 9},
    {"k": 3, "실루엣": 0.42, "가장 작은 그룹": 2350},
    {"k": 4, "실루엣": 0.39, "가장 작은 그룹": 1480},
    {"k": 5, "실루엣": 0.34, "가장 작은 그룹": 960},
])
f_table["가장 작은 그룹 비중"] = (f_table["가장 작은 그룹"] / 16000 * 100).round(2)
f_table

,k,실루엣,가장 작은 그룹,가장 작은 그룹 비중
0,2,0.61,9,0.06
1,3,0.42,2350,14.69
2,4,0.39,1480,9.25
3,5,0.34,960,6.00


## 3-1. F의 결과를 CRM팀에 그대로 넘겨도 될까?

**재현**: Online Retail II의 실제 고객으로 F와 같은 절차를 밟습니다. ⑴ 단위 그대로의 11개 피처 → PCA 2축 → K-means(k=2~5)를 돌려 **작은 그룹이 생기는지**, ⑵ PC1이 무엇으로 만들어지는지(규모 축인지), ⑶ 로그·표준화로 다시 하면 달라지는지, ⑷ 다음 해 행동으로 검증되는지를 차례로 봅니다.

In [9]:
# [재현] 고객 1명 = 1행, 11개 피처(단위 그대로)
raw = pd.read_csv("online_retail_II.csv", dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
sales = raw[raw["CustomerID"].notna() & ~raw["Invoice"].str.startswith("C")
            & (raw["Quantity"] > 0) & (raw["Price"] > 0)].copy()
sales["CustomerID"] = sales["CustomerID"].astype(int)
sales["amount"] = sales["Quantity"] * sales["Price"]
y1 = sales[(sales["InvoiceDate"] >= "2009-12-01") & (sales["InvoiceDate"] < "2010-12-01")]
y2 = sales[(sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2011-12-01")]

g = y1.groupby("CustomerID")
cust = pd.DataFrame({
    "orders": g["Invoice"].nunique(),
    "items": g["Quantity"].sum(),
    "spend": g["amount"].sum(),
    "avg_line": g["amount"].mean(),
    "max_line": g["amount"].max(),
    "distinct_products": g["StockCode"].nunique(),
    "active_days": g["InvoiceDate"].apply(lambda s: s.dt.date.nunique()),
    "uk_share": g["Country"].apply(lambda s: (s == "United Kingdom").mean()),
    "avg_qty": g["Quantity"].mean(),
    "cancel_like": g["amount"].apply(lambda s: (s <= 0).mean()),
    "recency": (pd.Timestamp("2010-12-01") - g["InvoiceDate"].max()).dt.days,
})
print("고객 수:", len(cust), "| 피처 수:", cust.shape[1])
cust.describe().round(1)

고객 수: 4266 | 피처 수: 11


,orders,items,spend,avg_line,max_line,distinct_products,active_days,uk_share,avg_qty,cancel_like,recency
count,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0,4266.0
mean,4.3,1261.1,1997.2,36.9,126.5,62.6,3.8,0.9,21.4,0.0,87.2
std,7.8,6403.5,8601.8,218.7,462.1,84.2,5.8,0.3,115.6,0.0,95.1
min,1.0,1.0,3.0,2.1,3.0,1.0,1.0,0.0,1.0,0.0,0.0
25%,1.0,157.2,307.5,11.2,31.8,17.0,1.0,1.0,5.3,0.0,15.0
50%,2.0,379.0,689.1,17.3,59.4,38.0,2.0,1.0,9.4,0.0,47.0
75%,5.0,977.8,1676.4,24.8,103.7,78.0,4.0,1.0,14.2,0.0,134.0
max,197.0,220600.0,321329.7,10953.5,15818.4,1716.0,123.0,1.0,4460.3,0.0,364.0


In [10]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

F_raw = cust.values
pca_raw = PCA(n_components=2).fit(F_raw)
print("단위 그대로 PCA: PC1", round(pca_raw.explained_variance_ratio_[0], 3),
      "| PC2", round(pca_raw.explained_variance_ratio_[1], 3))
print("PC1 가중치 (큰 순)")
print(pd.Series(pca_raw.components_[0], index=cust.columns).sort_values(ascending=False).round(3).head(6).to_string())

print()
print("[단위 그대로 PCA 2축 → K-means]")
rows = []
for k in range(2, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(pca_raw.transform(F_raw))
    sizes = np.bincount(km.labels_)
    rows.append({"k": k, "실루엣": round(silhouette_score(pca_raw.transform(F_raw), km.labels_), 3),
                 "가장 작은 그룹": int(sizes.min()), "그룹 크기": sizes.tolist()})
    print(f"  k={k}: 실루엣 {rows[-1]['실루엣']} | 가장 작은 그룹 {sizes.min()}명 | 크기 {sizes.tolist()}")
f_repro = pd.DataFrame(rows)

단위 그대로 PCA: PC1 0.896 | PC2 0.102
PC1 가중치 (큰 순)
spend                0.825
items                0.565
max_line             0.018
distinct_products    0.003
avg_qty              0.003
avg_line             0.002

[단위 그대로 PCA 2축 → K-means]


  File "/workspace/.local/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 272, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


  k=2: 실루엣 0.987 | 가장 작은 그룹 6명 | 크기 [4260, 6]


  k=3: 실루엣 0.959 | 가장 작은 그룹 5명 | 크기 [4235, 5, 26]


  k=4: 실루엣 0.953 | 가장 작은 그룹 2명 | 크기 [4233, 2, 4, 27]


  k=5: 실루엣 0.868 | 가장 작은 그룹 3명 | 크기 [4118, 3, 18, 3, 124]


In [11]:
# 로그 + 표준화로 다시: 규모 축을 걷어 내고 '행동'의 축을 본다
from sklearn.preprocessing import StandardScaler
X_log = StandardScaler().fit_transform(np.log1p(cust.clip(lower=0)))
pca_log = PCA(n_components=2).fit(X_log)
print("로그+표준화 PCA: PC1", round(pca_log.explained_variance_ratio_[0], 3),
      "| PC2", round(pca_log.explained_variance_ratio_[1], 3))
print("PC1 가중치")
print(pd.Series(pca_log.components_[0], index=cust.columns).sort_values(ascending=False).round(3).to_string())
print()
rows = []
for k in range(2, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_log)
    sizes = np.bincount(km.labels_)
    rows.append({"k": k, "실루엣": round(silhouette_score(X_log, km.labels_), 3),
                 "가장 작은 그룹": int(sizes.min()), "그룹 크기": sizes.tolist()})
    print(f"  k={k}: 실루엣 {rows[-1]['실루엣']} | 가장 작은 그룹 {sizes.min()}명 | 크기 {sizes.tolist()}")
f_log = pd.DataFrame(rows)

로그+표준화 PCA: PC1 0.477 | PC2 0.248
PC1 가중치
spend                0.439
items                0.418
orders               0.414
active_days          0.411
max_line             0.308
distinct_products    0.304
avg_qty              0.160
avg_line             0.125
cancel_like         -0.000
uk_share            -0.034
recency             -0.249



  k=2: 실루엣 0.295 | 가장 작은 그룹 1761명 | 크기 [2505, 1761]


  k=3: 실루엣 0.29 | 가장 작은 그룹 477명 | 크기 [477, 2258, 1531]


  k=4: 실루엣 0.305 | 가장 작은 그룹 316명 | 크기 [2004, 1405, 316, 541]


  k=5: 실루엣 0.243 | 가장 작은 그룹 298명 | 크기 [639, 1489, 1412, 298, 428]


In [12]:
# 안정성(초기값)과 검증(다음 해 행동)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_log)
aris = [adjusted_rand_score(km4.labels_, KMeans(n_clusters=4, n_init=10, random_state=s).fit(X_log).labels_) for s in range(5)]
print("k=4 초기값 5회 ARI:", np.round(aris, 3), "| 최저", round(min(aris), 3))

cust["k4"] = km4.labels_
next_spend = y2.groupby("CustomerID")["amount"].sum().reindex(cust.index).fillna(0)
cust["next_spend"] = next_spend
cust["repurchase"] = (cust["next_spend"] > 0).astype(int)
prof = cust.groupby("k4").agg(고객수=("spend", "size"), 구매액_중앙값=("spend", "median"),
                              주문수_중앙값=("orders", "median"), 다음해_재구매율=("repurchase", "mean"),
                              다음해_구매액_중앙값=("next_spend", "median"))
prof["다음해_재구매율"] = prof["다음해_재구매율"].round(3)
print()
print("전체 다음 해 재구매율:", round(cust["repurchase"].mean(), 3))
prof

k=4 초기값 5회 ARI: [0.886 0.922 0.425 1.    0.886] | 최저 0.425



전체 다음 해 재구매율: 0.636


,고객수,구매액_중앙값,주문수_중앙값,다음해_재구매율,다음해_구매액_중앙값
k4,,,,,
0,2004,326.645,1.0,0.476,0.00
1,1405,2029.340,6.0,0.875,1402.02
2,316,978.060,2.0,0.674,575.74
3,541,785.250,2.0,0.588,270.90


In [13]:
# PC2가 '육아 가구'인지 확인: 군집에 쓰지 않은 정보(품목 카테고리)를 쓴다
def share_of(keyword):
    m = y2["Description"].fillna("").str.contains(keyword, case=False, regex=True)
    s = y2[m].groupby("CustomerID")["amount"].sum().reindex(cust.index).fillna(0)
    return (s / cust["spend"].replace(0, np.nan)).fillna(0)

cust["baby_share"] = share_of("BABY|CHILD|KID|TOY|PLAYHOUSE|PUSHCHAIR|BOTTLE")
cust["home_share"] = share_of("HOME|KITCHEN|CAKESTAND|JAR|HOLDER|FRAME")
pc2 = pca_log.transform(X_log)[:, 1]
cust["PC2"] = pc2
q = pd.qcut(cust["PC2"], 4, labels=["PC2 하위", "중하", "중상", "PC2 상위"])
print("[PC2 사분위별 '유아·아동 상품 구매 비중'과 다음 해 재구매율]")
print(cust.groupby(q, observed=True).agg(고객수=("spend", "size"), 유아아동_비중=("baby_share", "mean"),
                                         홈_비중=("home_share", "mean"), 다음해_재구매율=("repurchase", "mean")).round(3))
print()
print("PC2와 유아·아동 비중의 상관:", round(np.corrcoef(cust["PC2"], cust["baby_share"])[0, 1], 3))

[PC2 사분위별 '유아·아동 상품 구매 비중'과 다음 해 재구매율]
         고객수  유아아동_비중   홈_비중  다음해_재구매율
PC2                                   
PC2 하위  1067    0.046  0.292     0.691
중하      1066    0.048  0.103     0.704
중상      1066    0.049  0.117     0.580
PC2 상위  1067    0.048  0.104     0.570

PC2와 유아·아동 비중의 상관: 0.013


## 답 3-1. 그대로 넘기지 않는다 — k=2는 "9명짜리 이상치 그룹"이고, PC1은 구매력이 아니라 규모다

**1. F의 결과에서 무엇이 문제인가**
- **k=2의 실루엣 0.61은 9명짜리 그룹이 만든 착시입니다.** 16,000명 중 9명(0.06%)을 따로 떼어 놓으면 "잘 나뉘었다"는 점수가 올라가지만, CRM팀이 9명에게 별도 정책을 만들 수는 없습니다. 실루엣이 **가장 높은 k를 고르는 것**이 목적이 아닙니다.
- **PC1(61%)은 "구매력"이 아니라 "총 구매액(규모) 그 자체"입니다.** 피처를 단위 그대로 넣으면 분산이 큰 변수(총 구매액)가 축을 독점합니다. 같은 절차를 Online Retail II 4,266명에 적용해 보니 **PC1이 89.6%를 설명**했고 가중치는 **총 구매액 0.825**로 압도적이었습니다. F의 61%도 같은 성질로, "많이 산 고객 vs 적게 산 고객"을 축이라고 부른 것입니다.
- **단위 그대로의 K-means는 크기 순위로만 나눕니다.** 재현에서 k=2~5의 가장 작은 그룹이 **6·5·2·3명**이었고 실루엣은 0.99까지 치솟았습니다. 큰 고객 몇 명과 나머지 전원으로 갈린 결과입니다.
- **안정성 검증이 없습니다.** 재현에서 초기값만 5번 바꿔도 ARI가 **0.425~1.000**으로 흔들렸습니다. 이 상태로 그룹을 넘기면, 다음 달 재학습에서 고객의 그룹이 바뀌어 정책이 흔들립니다(의뢰서의 "자주 바뀌면 곤란" 위반).
- **외부 검증이 없습니다.** F의 보고에는 "그 그룹이 다음 기간에 실제로 다르게 행동했는가"가 없습니다.

**2. 그래서 이렇게 다시 하겠습니다**
1. **규모를 걷어 냅니다**: 로그 변환 + 표준화, 또는 규모 변수(총 구매액·총 수량)를 빼고 **행동 비율**(구매 빈도·최근성·상품 다양성·주문당 금액)로 만듭니다. 재현에서 이렇게 하자 그룹 크기가 **316~2,505명**으로 균형을 이루고, 실루엣은 0.24~0.31로 낮아졌지만 **그룹이 쓸모 있어졌습니다**.
2. **k를 네 기준으로 정합니다**: 실루엣(후보 좁히기) + **최소 그룹 크기(예: 5% 이상)** + 업무 제약(5개 이내) + **안정성(초기값·표본 변화에서 ARI 0.8 이상)**.
3. **쓰지 않은 기간으로 검증합니다**: 재현에서 k=4 그룹의 **다음 해 재구매율이 47.6% vs 87.5%**로 갈렸습니다. 이런 외부 검증이 있어야 "믿을 만하다"고 말할 수 있습니다.
4. **9명 그룹은 고객 그룹이 아니라 '이상 거래처' 목록으로 분리**합니다(Chapter 7의 이상 탐지로 넘깁니다).
5. **보고 문구를 바꿉니다**: "고객이 본래 4개 유형" → **"우리가 로그·표준화와 k=4를 골라 4개 그룹으로 나눴고, 다음 해 행동에서 차이를 확인했다"**. 방법을 바꾸면 결과가 달라진다는 단서(ARI 0.43~1.0)도 함께 적습니다.

> **한 줄 결정**: "F의 결과는 **반려**하고, 규모 변수를 걷어 낸 재분석(로그+표준화, k=4, 최소 그룹 5% 이상, 다음 해 검증)을 요청한다. 9명 그룹은 이상 거래처 목록으로 넘긴다."

## 답 3-2. "육아 가구 축"은 군집에 쓰지 않은 정보로만 확인할 수 있다

**1. 확인 방법 (군집에 쓰지 않은 정보)**
축의 이름은 **군집에 넣지 않은 변수**로 검증해야 합니다. 넣은 변수로 확인하면 순환 논리입니다.
- **유아·아동 상품 구매 비중**: 카테고리(유아복·장난감·기저귀·유모차 등) 구매액 비율, 유아 브랜드 구매 여부
- **반복 구매 성격**: 기저귀·분유 같은 소모품의 재구매 주기, 벌크 구매 비율
- **고객 정보**: 배송지 유형(주거), 가구원 수·자녀 유무(설문·회원 정보), 유아용품 검색·장바구니 담기 기록
- **행동 검증**: PC2 상위 그룹이 다음 기간에 유아용품을 **더 사는지**(재구매율·구매액)

절차는 ⑴ PC2 상위·하위 그룹의 유아용품 비중을 비교(비율 검정), ⑵ PC2 점수와 유아용품 비중의 상관, ⑶ 다음 기간 유아용품 구매로 외부 검증, ⑷ 표본 100명의 실제 확인(설문·상담 기록)입니다.

**2. 재현 결과: 이름이 확인되지 않았습니다**
Online Retail II에서 PC2 사분위별 **유아·아동 상품 구매 비중**은 0.046 / 0.048 / 0.049 / 0.048로 **차이가 없었고, PC2와의 상관은 0.013**이었습니다. 반면 **홈·주방 상품 비중**은 PC2 하위 그룹에서 **0.292**, 나머지에서 0.10~0.12로 차이가 났습니다. 즉 이 데이터의 PC2는 "육아 가구"가 아니라 **"홈·주방 상품 중심 vs 그 외"** 로 읽히는 축입니다. F가 붙인 "육아 가구"라는 이름도 **같은 방식으로 확인되지 않았다면 이름일 뿐**입니다.

**3. 보고서에 어떻게 적나**
- 확인되었을 때: "PC2(22%)는 유아·식품·생활용품에 함께 걸린 축이다. 유아·아동 상품 구매 비중이 PC2 상위 그룹에서 **△△%p 높고(검정 p<0.01)**, 다음 해 유아용품 재구매율도 높아 **'육아 가구'로 부를 수 있다**."
- 확인되지 않았을 때(이번 재현처럼): "PC2는 유아·식품·생활용품에 함께 걸린 축이지만, **유아용품 구매 비중으로는 확인되지 않았다(상관 0.013)**. 따라서 '생활용품 중심 가구(미검증)'로 표기하고, 이름을 확정하려면 회원 정보·카테고리 데이터가 필요하다."

**4. 추가로 필요한 정보 (CRM팀에 요청)**
⑴ 상품 카테고리 코드(유아·식품·생활용품 구분), ⑵ 회원 가입 시 자녀 유무·연령대, ⑶ 배송지·가구원 수, ⑷ 검색·장바구니 로그, ⑸ 설문 또는 상담 기록. 이 정보가 있으면 "육아 가구"라는 이름을 **검증된 세그먼트**로 쓸 수 있고, 없으면 **"미검증 축"으로 남겨 두어야** 합니다.

---
# 마무리 — 세 문제를 관통하는 판단 원칙

| 문제 | 결정 | 근거로 삼은 숫자 | 하지 않은 일 |
| --- | --- | --- | --- |
| 1. 부스팅 전환 | (아래 답 참조) | fold별 차이·폭, 500명 기준 차이 | 검증 표의 최고값만 보고 결정 |
| 2. 보고서·캠페인 | (아래 답 참조) | 기여도 방향·상관, 손익분기 이탈 감소율 | 인과 단정, SHAP을 개입 효과로 읽기 |
| 3. 군집 인계 | (아래 답 참조) | 가장 작은 그룹 크기, 실루엣 vs 쓸모, 다음 해 검증 | 실루엣 최고 k를 그대로 채택 |

## 제출 체크리스트

- [x] 문제 1: 학습/검증 격차와 앙상블의 효과를 **숫자로** 설명
- [x] 문제 1: 전환 여부를 **비용·이득과 오차 범위**로 판단
- [x] 문제 2: 틀린 문장을 찾아 **이유와 수정 문장** 제시
- [x] 문제 2: 1.6억 원 캠페인을 **손익분기**로 판단
- [x] 문제 3: F의 결과를 넘길지, 무엇을 다시 할지 제시
- [x] 문제 3: "육아 가구 축"을 **쓰지 않은 정보**로 확인하는 방법 제시